In [13]:
import os
import re
import mysql.connector
from mysql.connector import Error

# ===========================
# CONFIGURATION
# ===========================
USER = "niraj"
PASSWORD = "Niraj@123"
HOST = "localhost"
PORT = 3306
DATABASE = "drugbank"

# Path to your SQL schema file
SQL_FILE = "DrugBank_MetaData.sql" 

# ===========================
# EXECUTION NODE
# ===========================
conn = None
cursor = None

try:
    # 1. Connect directly to MySQL database
    conn = mysql.connector.connect(
        host=HOST,
        port=PORT,
        user=USER,
        password=PASSWORD,
        database=DATABASE,
        autocommit=True
    )
    cursor = conn.cursor()
    print(f"[SUCCESS] Connected to MySQL database '{DATABASE}' as user '{USER}'.")

    # 2. Check SQL Schema File Existence
    if not os.path.exists(SQL_FILE):
        raise FileNotFoundError(f"File '{SQL_FILE}' not found in working directory.")

    print(f"[INFO] Reading SQL dump file '{SQL_FILE}'...")
    with open(SQL_FILE, "r", encoding="utf-8") as f:
        sql_script = f.read()

    # 3. Disable Foreign Key Checks during schema rebuild
    cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
    print("[INFO] Foreign key checks temporarily disabled for clean table creation.")

    # 4. Parse and Execute Statements Cleanly
    # Remove single-line comments (-- ...)
    clean_script = re.sub(r'--.*?\n', '\n', sql_script)
    
    statements = clean_script.split(";")
    success = 0
    failed = 0

    for statement in statements:
        stmt = statement.strip()
        # Skip empty lines, transaction keywords, or phpMyAdmin header comments
        if not stmt or stmt.upper() in ["START TRANSACTION", "COMMIT"] or stmt.startswith("/*"):
            continue
            
        try:
            cursor.execute(stmt)
            success += 1
        except Error as err:
            # Ignore minor non-blocking conditional comment errors
            if "1065" in str(err) or "1064" in str(err):
                continue
            failed += 1
            print("=" * 60)
            print("Failed Statement Preview:")
            print(stmt[:200] + "...")
            print("Error Details:", err)
            print("=" * 60)

    # 5. Re-enable Foreign Key Checks
    cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")

    print("\n========================================")
    print("Schema Execution Summary")
    print(f"Successful statements : {success}")
    print(f"Failed statements     : {failed}")
    print("========================================")

    # 6. Enforce AUTO_INCREMENT on drug_pk (Required for Phase 1 / XML Ingestion)
    print("\n[INFO] Enforcing AUTO_INCREMENT & UNIQUE KEY constraint on drug.drug_pk...")
    try:
        # First check if uq_drug_pk already exists to avoid duplicate key name error
        cursor.execute("SHOW INDEX FROM drug WHERE Key_name = 'uq_drug_pk'")
        if not cursor.fetchone():
            cursor.execute("ALTER TABLE drug MODIFY drug_pk BIGINT NOT NULL AUTO_INCREMENT, ADD UNIQUE KEY uq_drug_pk (drug_pk);")
        else:
            cursor.execute("ALTER TABLE drug MODIFY drug_pk BIGINT NOT NULL AUTO_INCREMENT;")
        print("[SUCCESS] Table 'drug' schema successfully configured for auto-incrementing PK.")
    except Error as pk_err:
        print(f"[NOTE] PK Alter notice: {pk_err}")

except Error as e:
    print("Connection Error:", e)

finally:
    if cursor:
        try:
            cursor.close()
        except Exception:
            pass
    if conn:
        try:
            conn.close()
        except Exception:
            pass

print("\n[DONE] Database setup complete. You can now execute Stage 1 XML loading.")

[SUCCESS] Connected to MySQL database 'drugbank' as user 'niraj'.
[INFO] Reading SQL dump file 'DrugBank_MetaData.sql'...
[INFO] Foreign key checks temporarily disabled for clean table creation.

Schema Execution Summary
Successful statements : 131
Failed statements     : 0

[INFO] Enforcing AUTO_INCREMENT & UNIQUE KEY constraint on drug.drug_pk...
[SUCCESS] Table 'drug' schema successfully configured for auto-incrementing PK.

[DONE] Database setup complete. You can now execute Stage 1 XML loading.


In [11]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="niraj",
    password="Niraj@123",
    database="drugbank"
)

cursor = conn.cursor()

cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

print(f"Total Tables: {len(tables)}")

for table in tables:
    print(table[0])

cursor.close()
conn.close()

Total Tables: 43
drug
drug_affected_organism
drug_ahfs_code
drug_atc_code
drug_atc_level
drug_category
drug_classification
drug_classification_alternative_parent
drug_classification_substituent
drug_dosage
drug_external_identifier
drug_external_link
drug_food_interaction
drug_group
drug_interaction
drug_patent
drug_pathway
drug_pdb_entry
drug_price
drug_property
drug_raw
drug_reference
drug_salt
drug_salt_drugbank_id
drug_synonym
drugbank_id_map
interactant
interactant_action
interactant_polypeptide
interactant_reference
manufacturer
packager
pathway
pathway_drug_member
pathway_enzyme_member
polypeptide
polypeptide_external_identifier
polypeptide_synonym
product
reaction
reaction_element
reaction_enzyme
reference_item
